In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import logging
logging.basicConfig(level=logging.ERROR, format='%(levelname)s: %(message)s')

import os
import sys
import pandas as pd
import numpy as np
import tensorflow as tf # type: ignore

from meridian.model import model
from meridian.model import spec
from meridian.analysis import optimizer
from meridian.analysis import analyzer

from meridian.planner.flex_budget_planner import FlexibleBudgetPlanner, CompareOptimizedVsNonOptimized
from meridian.analysis.optimizer import OptimizationResults


In [3]:
# Optimizer input excel file path
home_dir = '/Users/mariappan.subramanian/Library/CloudStorage/OneDrive-TheTradeDesk/MMM/BudgetOptimizer'
input_file_path = f'{home_dir}/input_files/optimizer_input_case_coeff.xlsx'
if not os.path.exists(input_file_path):
  raise FileNotFoundError(f'File not found: {input_file_path}')

In [5]:
# Configuration for input excel file
input_config = {

  # time and geo inputs
  'time_col': 'week',
  'geo_col': 'geo',
  'population_col': 'population',

  # kpi inputs
  'kpi_col': 'conversions',  #
  'kpi_type': 'non_revenue',
  'revenue_per_kpi_col': 'revenue_per_conversion',  # needed if kpi_type is non_revenue

  # impression based media inputs
  'media_cols': ['Channel0_impression', 'Channel1_impression', 'Channel2_impression'],
  'media_spend_cols': ['Channel0_spend', 'Channel1_spend', 'Channel2_spend'],
  'media_channels': ['Channel0', 'Channel1', 'Channel2'],

  # rf inputs
  'reach_cols': ['Channel3_reach'],
  'frequency_cols': ['Channel3_frequency'],
  'rf_spend_cols': ['Channel3_spend'],
  'rf_channels': ['Channel3']

}

# optimization config
optimization_config = {
  'fixed_budget': True
}

In [6]:
# call the planner
planner = FlexibleBudgetPlanner(file_name=input_file_path, model_config=input_config)
opt_results = planner.optimize(optimization_config)

I0000 00:00:1759338730.603731 3316191 service.cc:148] XLA service 0x105ce3b60 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1759338730.603760 3316191 service.cc:156]   StreamExecutor device (0): Host, Default Version
I0000 00:00:1759338730.611915 3316191 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, decay_function, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


2025-10-01 12:12:11.310108: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.


In [8]:
# summarize optimized vs non-optimized
compare_opt_vs_nonopt = CompareOptimizedVsNonOptimized(opt_results)
total_opt_vs_nonopt_df = compare_opt_vs_nonopt.get_total_level_comparison()
channel_opt_vs_nonopt_df = compare_opt_vs_nonopt.get_channel_level_comparison()

In [9]:
total_opt_vs_nonopt_df

,start_date,end_date,optimized_budget,optimized_total_incremental_outcome,optimized_total_cpa,nonoptimized_budget,nonoptimized_total_incremental_outcome,nonoptimized_total_cpa,budget_change,outcome_change,cpa_change
0,2021-01-25,2024-01-15,113300000.0,348625088.0,0.324991,110900000.0,309880960.0,0.357879,0.021641,0.125029,-0.091898


In [10]:
channel_opt_vs_nonopt_df

,channel,optimized_spend,optimized_incremental_outcome,optimized_effectiveness,optimized_cpa,nonoptimized_spend,nonoptimized_incremental_outcome,nonoptimized_effectiveness,nonoptimized_cpa,budget_change,outcome_change,effectiveness_change,cpa_change
0,Channel0,28300000,51791016.0,0.020347,0.546427,40400000,65176604.0,0.017936,0.619854,-0.299505,-0.205374,0.134377,-0.118459
1,Channel1,31300000,73407784.0,0.026815,0.426385,27600000,68637496.0,0.028434,0.402113,0.134058,0.069500,-0.056927,0.060363
2,Channel2,28200000,85158880.0,0.036063,0.331146,23300000,77261024.0,0.039600,0.301575,0.210300,0.102223,-0.089298,0.098054
3,Channel3,25500000,138267424.0,0.063769,0.184425,19600000,98805824.0,0.059287,0.198369,0.301020,0.399385,0.075606,-0.070292


In [11]:
display(opt_results.plot_response_curves())
display(opt_results.plot_incremental_outcome_delta())
display(opt_results.plot_spend_delta())

alt.FacetChart(...)

alt.LayerChart(...)

alt.LayerChart(...)